# Frozen learned policy: 300 held-out questions

Freeze all selected checkpoint hashes and the validation-selected static policy before accessing held-out model results. Apply the learned policy deterministically. Compare NoComm, self-update, selected static, quota-limited broadcast and per-seed random supports matched on learned edge count/quota. Random supports do not guarantee identical recipient participation or actual payload identities, so this is a whole-policy comparison; use E2 for destination-only claims.

Report F1, EM, evidence recall, actual payload, full provider input/output usage and actual recipient call counts. Report each optimizer seed and seed-averaged question-paired contrasts, including learned versus matched random, with uncertainty and exploratory Holm correction. Never count 300×3 as 900 independent questions.

Action/NoComm frequencies expose policy collapse. A null, negative or NoComm-dominant result is retained. Three hundred questions is a feasible target, not a guarantee of sufficient power or reviewer approval. Do not inspect held-out results and then retrain, tune or enlarge this test set.

Active sample-size revision: **100 fixed training questions, 30 fixed validation questions, and 300 unchanged separate held-out questions**. Three collection epochs and three optimizer seeds remain. Previous 300/100 artifacts are preserved as a separate study; their predictions and scores did not select the subsets. Reused provider responses retain their original provenance and costs. This notebook has not yet executed the new study to completion.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'protocol.py').exists() else Path.cwd() / 'WEBExtension'
if not (ROOT / 'protocol.py').exists():
    raise FileNotFoundError('Launch Jupyter from the project root or WEBExtension folder')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
from protocol import load_official, write_json, file_hash
from experiments import load_config, preview, freeze, run
CFG = load_config(ROOT / 'config.json')
EXECUTE = False  # Set True only after configuring your real backend and maximum call budget.
print('Notebook root:', ROOT)
print('Inference enabled:', EXECUTE)

SETTINGS=json.loads((ROOT/'controller_config.json').read_text())
from controller_study import forecast, study_directory
STUDY_DIR,SPLIT_DIR=study_directory(CFG,SETTINGS)
print(json.dumps(forecast(CFG,SETTINGS),indent=2))


Notebook root: /home/saurab/Documents/projectRLMAIN/WEBExtension
Inference enabled: False


{
  "study_dir": "/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_study_a0b3768357484b45",
  "split_dir": "/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_splits_2e025f8ae3d9815e",
  "train": 100,
  "validation": 30,
  "heldout": 300,
  "seeds": [
    7,
    19,
    31
  ],
  "conservative_max_calls_before_reuse": 26340,
  "configured_operational_cap": 46000,
  "note": "Upper bound; identical initial and receiver prompts reuse real cached responses. Pilot costs are separate."
}


In [2]:
from controller_study import evaluate
print(evaluate(CFG,SETTINGS,execute=EXECUTE))

{'study_dir': '/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_study_a0b3768357484b45', 'split_dir': '/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_splits_2e025f8ae3d9815e', 'train': 100, 'validation': 30, 'heldout': 300, 'seeds': [7, 19, 31], 'conservative_max_calls_before_reuse': 26340, 'configured_operational_cap': 46000, 'note': 'Upper bound; identical initial and receiver prompts reuse real cached responses. Pilot costs are separate.'}


In [3]:
from controller_reporting import report_controller
from IPython.display import display,Image
status=STUDY_DIR/'E5_status.json'
if status.exists() and json.loads(status.read_text()).get('status')=='COMPLETE':
    summary,statistics=report_controller(STUDY_DIR)
    display(summary)
    print(json.dumps(statistics,indent=2))
    display(Image(filename=str(STUDY_DIR/'reports/E5_payload_frontier.png')))
else:
    print('NO COMPLETE HELD-OUT CONTROLLER RESULTS. No table or trained-policy claim is fabricated.')

NO COMPLETE HELD-OUT CONTROLLER RESULTS. No table or trained-policy claim is fabricated.
